# 03 PDF CV Extraction

This notebook tests text extraction from a real CV in PDF format.

Main goals:
- load a PDF CV from the local project folder
- extract text using `pypdf`
- extract text using `pdfplumber`
- compare extraction quality
- clean extracted CV text
- save the final extracted text for later CV quality analysis and structured extraction

## 1. Imports and Settings

In [1]:
import re
import pandas as pd

from pathlib import Path
from pypdf import PdfReader
import pdfplumber

In [2]:
RAW_TEST_CV_DIR = Path("../data/raw/test_cv")
PROCESSED_TEST_CV_DIR = Path("../data/processed/test_cv")

test_cv="Marko Mijanovic CV.pdf"

#test_cv="CV-ks.pdf"

In [3]:
cv_path=RAW_TEST_CV_DIR/test_cv
print(cv_path)

..\data\raw\test_cv\CV-ks.pdf


## 2. Extract Text Using pypdf

`pypdf` is a simple PDF text extraction library.  
It works well for many standard PDFs, but may struggle with complex layouts, tables or multi-column CVs.

In [4]:
def extract_text_with_pypdf(pdf_path):
    
    reader = PdfReader(pdf_path)
    extracted_pages = []

    for page_number, page in enumerate(reader.pages, start=1):
        page_text = page.extract_text()


        if page_text is None:
            page_text = ""

        extracted_pages.append(page_text)

    full_text = "\n\n".join(extracted_pages)

    return full_text

In [5]:
pypdf_text=extract_text_with_pypdf(cv_path)

In [6]:
print(pypdf_text)

A L E K S A N D A R
B R A J I Ć
S T U D E N T  A T  T H E  F A C U L T Y  O F
O R G A N I Z A T I O N A L  S C I E N C E S
+ 3 8 1  6 1  6 4 2  7 4  2 4  
B u l e v a r  o s l o b o đ e n j a  3 7 9 b ,  B e o g r a d
b r a j i c a 2 0 0 2 @ g m a i l . c o m                        
i n / a l e k s a n d a r - b r a j i c
A  c o m m u n i c a t i v e  a n d  a d a p t a b l e  s t u d e n t ,  e a g e r  t o  l e a r n  a n d
d e v e l o p  n e w  s k i l l s .  P a s s i o n a t e  a b o u t  t e c h n o l o g y  a n d
p r o b l e m - s o l v i n g ,  w i t h  a  p r o a c t i v e  a p p r o a c h  t o  c h a l l e n g e s .
A l w a y s  o p e n  t o  n e w  o p p o r t u n i t i e s  f o r  g r o w t h  a n d  i m p r o v e m e n t .
J a v a
C #
H T M L
J a v a  S c r i p t
S e r b i a n  -  n a t i v e
E n g l i s h  -  B 2  ( a c t i v e l y  i m p r o v i n g  t h r o u g h  c o u r s e s )
I t a l i a n  -  A 2 / B 1
S K I L L S
L A N G U A G E S
E D U C A T I O N
- B a c h e l o

## 3. Extract Text Using pdfplumber

`pdfplumber` is often better for CVs because it can preserve layout and line structure more accurately than basic PDF extractors.

In [7]:
def extract_text_with_pdfplumber(pdf_path):

    extracted_pages = []


    with pdfplumber.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf.pages, start=1):
            page_text = page.extract_text()

            if page_text is None:
                page_text = ""

            extracted_pages.append(page_text)

    full_text = "\n\n".join(extracted_pages)

    return full_text

In [8]:
pdfplumber_text=extract_text_with_pdfplumber(cv_path)

In [9]:
print(pdfplumber_text)

A L E K S A N D A R
B R A J I Ć
STUDENT AT THE FACULTY OF
ORGANIZATIONAL SCIENCES
A communicative and adaptable student, eager to learn and
develop new skills. Passionate about technology and
problem-solving, with a proactive approach to challenges.
Always open to new opportunities for growth and improvement.
EXPERIENCE
DATA ENGINEERING INTERN
Levi9 Technology Services – Belgrade
Nov. 2025 – Dec. 2025 (3-week internship)
Developed and maintained automated data pipelines using
+381 61 642 74 24 Python, PySpark, and AWS to support analytics.
Worked with AWS services (S3, DynamoDB, AWS Glue, SQS) for
brajica2002@gmail.com
data ingestion, processing, and reliability.
Monitored data pipelines and system performance using Grafana
Bulevar oslobođenja 379b, Beograd
and CloudWatch.
Collaborated in an Agile/Scrum environment, working closely
in/aleksandar-brajic
with cross-functional teams.
github.com/Aleksandar169
IT INTERN
Systrax Technology - Trebinje
May 2025
EDUCATION
Generated accounting r

## 4. Compare Extraction Results

Both extraction methods are compared using basic text statistics.

The final method should be selected based on:
- text length
- readability
- preservation of sections
- whether skills, education and experience are visible

In [12]:
comparison = pd.DataFrame({
    "method": ["pypdf", "pdfplumber"],
    "character_count": [len(pypdf_text), len(pdfplumber_text)],
    "word_count": [len(pypdf_text.split()), len(pdfplumber_text.split())],
    "line_count": [len(pypdf_text.splitlines()), len(pdfplumber_text.splitlines())]
})



In [13]:
comparison

,method,character_count,word_count,line_count
0,pypdf,3489,1594,70
1,pdfplumber,1858,265,56


Although `pypdf` extracted a higher number of characters and words, the `pdfplumber` output was selected as the final extraction method because it produced a more readable and better structured CV text. Since all relevant CV information was preserved, readability and structure were prioritized over raw text length.

In [14]:
 raw_cv_text = pdfplumber_text

## 5. Clean Extracted CV Text

In [15]:
def clean_extracted_cv_text(text):
 
    if text is None:
        return ""

    text = str(text)

    text = text.replace("\x00", " ")

    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    lines = text.split("\n")
    cleaned_lines = []

    for line in lines:

        line = re.sub(r"[ \t]+", " ", line)

        line = line.strip()

        cleaned_lines.append(line)

    text = "\n".join(cleaned_lines)

    text = re.sub(r"\n{3,}", "\n\n", text)

    text = text.strip()

    return text

In [16]:
clean_cv_text = clean_extracted_cv_text(raw_cv_text)


In [17]:

print(clean_cv_text)

A L E K S A N D A R
B R A J I Ć
STUDENT AT THE FACULTY OF
ORGANIZATIONAL SCIENCES
A communicative and adaptable student, eager to learn and
develop new skills. Passionate about technology and
problem-solving, with a proactive approach to challenges.
Always open to new opportunities for growth and improvement.
EXPERIENCE
DATA ENGINEERING INTERN
Levi9 Technology Services – Belgrade
Nov. 2025 – Dec. 2025 (3-week internship)
Developed and maintained automated data pipelines using
+381 61 642 74 24 Python, PySpark, and AWS to support analytics.
Worked with AWS services (S3, DynamoDB, AWS Glue, SQS) for
brajica2002@gmail.com
data ingestion, processing, and reliability.
Monitored data pipelines and system performance using Grafana
Bulevar oslobođenja 379b, Beograd
and CloudWatch.
Collaborated in an Agile/Scrum environment, working closely
in/aleksandar-brajic
with cross-functional teams.
github.com/Aleksandar169
IT INTERN
Systrax Technology - Trebinje
May 2025
EDUCATION
Generated accounting r

In [18]:
pukni

NameError: name 'pukni' is not defined

## 6. Save Extracted CV Text

In [ ]:
output_txt_path = PROCESSED_TEST_CV_DIR / "test_cv_extracted_text.txt"

with open(output_txt_path, "w", encoding="utf-8") as file:
    file.write(clean_cv_text)

print(f"Extracted CV text saved to: {output_txt_path}")